# Student Performance Tracker

This notebook walks through the full machine learning workflow for predicting a student's final exam score:

1. Data Cleaning
2. Exploratory Data Analysis (EDA)
3. Correlation Matrix
4. Feature Engineering
5. Train/Test Split
6. Model Training (Linear Regression, Random Forest, Gradient Boosting)
7. Model Evaluation & Comparison (MAE, RMSE, R2)
8. Saving the Best Model with Pickle

**Author:** CodeTech IT Solutions Internship -- Project 1


In [ ]:
# 1. Imports
import sys
import os
sys.path.append(os.path.join(os.getcwd(), "..", "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import pickle

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)

# Local helper functions (shared with app.py / train_models.py)
from data_preprocessing import load_data, clean_data, engineer_features, get_feature_target_split

## 1. Load the Raw Dataset

In [ ]:
# Load the sample dataset
df_raw = load_data("../data/student_performance.csv")

print("Shape:", df_raw.shape)
df_raw.head()

In [ ]:
# Quick look at data types and missing values
df_raw.info()
print("\nMissing values per column:")
print(df_raw.isna().sum())

## 2. Data Cleaning

We fill missing numeric values with the median, strip whitespace from text columns, and drop exact duplicate rows.

In [ ]:
df_clean = clean_data(df_raw)

print("Missing values after cleaning:")
print(df_clean.isna().sum())
df_clean.head()

## 3. Exploratory Data Analysis (EDA)

In [ ]:
# Summary statistics
df_clean.describe()

In [ ]:
# Distribution of the target variable
plt.figure()
sns.histplot(df_clean["final_score"], kde=True, bins=25, color="steelblue")
plt.title("Distribution of Final Exam Scores")
plt.xlabel("Final Score")
plt.ylabel("Number of Students")
plt.show()

In [ ]:
# Study hours vs final score
plt.figure()
sns.scatterplot(data=df_clean, x="study_hours_per_week", y="final_score", hue="parental_support")
plt.title("Study Hours per Week vs Final Score")
plt.xlabel("Study Hours per Week")
plt.ylabel("Final Score")
plt.legend(title="Parental Support")
plt.show()

In [ ]:
# Attendance vs final score
plt.figure()
sns.scatterplot(data=df_clean, x="attendance_percent", y="final_score", hue="internet_access")
plt.title("Attendance (%) vs Final Score")
plt.xlabel("Attendance (%)")
plt.ylabel("Final Score")
plt.show()

In [ ]:
# Average final score by parental support level
avg_by_support = df_clean.groupby("parental_support")["final_score"].mean().reindex(["Low", "Medium", "High"])

plt.figure()
avg_by_support.plot(kind="bar", color=["#e74c3c", "#f1c40f", "#2ecc71"])
plt.title("Average Final Score by Parental Support Level")
plt.ylabel("Average Final Score")
plt.xticks(rotation=0)
plt.show()

## 4. Correlation Matrix

In [ ]:
numeric_df = df_clean.select_dtypes(include=[np.number])
corr = numeric_df.corr()

plt.figure(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", square=True, cbar_kws={"shrink": 0.8})
plt.title("Correlation Matrix of Numeric Features")
plt.show()

## 5. Feature Engineering

We derive a few additional features that may help the models capture non-linear relationships:

- `study_attendance_ratio`: study hours relative to attendance
- `engagement_score`: a blended measure of study time and attendance
- `rest_balance`: sleep hours adjusted for extracurricular load
- Ordinal encoding for `parental_support`
- One-hot encoding for `gender`, `internet_access`, and `part_time_job`


In [ ]:
df_processed = engineer_features(df_clean)
df_processed.head()

## 6. Train / Test Split

In [ ]:
X, y = get_feature_target_split(df_processed, target="final_score")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training set size:", X_train.shape)
print("Test set size:", X_test.shape)

## 7. Model Training

We train three regression models: Linear Regression, Random Forest Regressor, and Gradient Boosting Regressor.

In [ ]:
models = {
    "Linear Regression": LinearRegression(),
    "Random Forest Regressor": RandomForestRegressor(n_estimators=200, max_depth=8, random_state=42),
    "Gradient Boosting Regressor": GradientBoostingRegressor(n_estimators=200, learning_rate=0.05, max_depth=3, random_state=42),
}

fitted_models = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    fitted_models[name] = model
    print(f"Trained: {name}")

## 8. Model Evaluation & Comparison

In [ ]:
def evaluate_model(name, model, X_test, y_test):
    preds = model.predict(X_test)
    mae = mean_absolute_error(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    r2 = r2_score(y_test, preds)
    return {"Model": name, "MAE": mae, "RMSE": rmse, "R2": r2}

results = [evaluate_model(name, model, X_test, y_test) for name, model in fitted_models.items()]
results_df = pd.DataFrame(results).sort_values("R2", ascending=False).reset_index(drop=True)
results_df

In [ ]:
# Visual comparison of model performance
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for ax, metric in zip(axes, ["MAE", "RMSE", "R2"]):
    sns.barplot(data=results_df, x="Model", y=metric, ax=ax, palette="viridis")
    ax.set_title(metric)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=20, ha="right")

plt.tight_layout()
plt.show()

## 9. Save the Best Model

In [ ]:
best_model_name = results_df.iloc[0]["Model"]
best_model = fitted_models[best_model_name]
print(f"Best model: {best_model_name}")

os.makedirs("../models", exist_ok=True)
with open("../models/best_model.pkl", "wb") as f:
    pickle.dump({
        "model": best_model,
        "model_name": best_model_name,
        "feature_columns": list(X.columns),
    }, f)

results_df.to_csv("../models/model_comparison.csv", index=False)
print("Best model and comparison table saved to the 'models/' directory.")

## 10. Conclusion

The notebook above cleaned the raw student data, explored relationships between lifestyle/study
factors and exam performance, engineered new features, and compared three regression models.
The best-performing model (by R2) was saved to `models/best_model.pkl` and is used by the
Streamlit app (`app.py`) to serve live predictions.

See the project `README.md` for instructions on running the Streamlit app and interpreting results.
